# Treinamento de Detector de Deepfake v4 - FACE-SWAP DATASET

## Problema identificado:
O dataset **140k Real and Fake Faces** contém rostos **gerados por StyleGAN**, NÃO deepfakes de face-swap!

O modelo treinado nele detecta:
- ✅ Rostos 100% sintéticos (StyleGAN, DALL-E, Midjourney)
- ❌ Face-swap deepfakes (DeepFaceLab, FaceSwap, etc.)

## Solução:
Usar datasets com **deepfakes reais de face-swap**:

1. **FaceForensics++** - 1000 vídeos originais + 4000 manipulados (4 métodos)
2. **Celeb-DF** - Deepfakes de celebridades de alta qualidade
3. **DFDC** - Deepfake Detection Challenge (Facebook)
4. **DeeperForensics** - 60K vídeos com variações realistas

Este notebook usa FaceForensics++ e permite adicionar seus próprios deepfakes.

## 1. Setup

In [ ]:
!nvidia-smi
!pip install -q timm albumentations kaggle scikit-learn matplotlib seaborn opencv-python-headless facenet-pytorch

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
import cv2
import json
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, f1_score, fbeta_score, roc_curve
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Configuração

In [ ]:
class Config:
    # Paths
    OUTPUT_DIR = Path('/content/output')
    CUSTOM_FAKE_DIR = Path('/content/custom_fakes')  # Seus deepfakes!
    CUSTOM_REAL_DIR = Path('/content/custom_reals')  # Suas fotos reais!
    
    # Modelo
    MODEL_NAME = 'efficientnet_b0'
    PRETRAINED = True
    
    # Training
    EPOCHS = 30
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 5e-4
    WARMUP_EPOCHS = 3
    
    # Loss - PESO ALTO para fakes
    FOCAL_GAMMA = 3.0
    FOCAL_ALPHA = 0.8  # 4x mais peso para detectar fakes
    LABEL_SMOOTHING = 0.1
    
    # Augmentation
    USE_MIXUP = True
    MIXUP_ALPHA = 0.4
    
    IMG_SIZE = 224
    USE_AMP = True
    NUM_WORKERS = 4
    PATIENCE = 10
    
    # Threshold - otimizar para RECALL (detectar fakes)
    THRESHOLD_METRIC = 'f2'
    MIN_THRESHOLD = 0.15
    MAX_THRESHOLD = 0.50

config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config.CUSTOM_FAKE_DIR.mkdir(parents=True, exist_ok=True)
config.CUSTOM_REAL_DIR.mkdir(parents=True, exist_ok=True)

## 3. Download de Datasets de Face-Swap

### Opção A: FaceForensics++ (requer termo de uso)
Acesse: https://github.com/ondyari/FaceForensics

### Opção B: Kaggle - Deepfake Detection Challenge
Dataset mais acessível com deepfakes reais.

In [ ]:
# Configurar Kaggle
from google.colab import files

# uploaded = files.upload()  # Descomente para fazer upload do kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ 2>/dev/null || echo 'Faça upload do kaggle.json'
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# OPÇÃO 1: Dataset com frames de vídeos deepfake
# Este dataset tem FRAMES extraídos de vídeos deepfake reais
!kaggle datasets download -d manjilkarki/deepfake-and-real-images -p /content/data_dfr --unzip

# A estrutura será:
# /content/data_dfr/
#   ├── Dataset/
#   │   ├── Train/
#   │   │   ├── Fake/
#   │   │   └── Real/
#   │   └── Validation/
#   │       ├── Fake/
#   │       └── Real/

In [ ]:
# OPÇÃO 2: Dataset 140k (como complemento - rostos sintéticos)
# Descomente se quiser combinar com o outro dataset
# !kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data_140k --unzip

In [ ]:
# OPÇÃO 3: Celeb-DF v2 (deepfakes de alta qualidade)
# Precisa baixar manualmente de: https://github.com/yuezunli/celeb-deepfakeforensics
# !unzip celeb-df-v2.zip -d /content/data_celebdf

## 4. Carregar e Verificar Dados

In [ ]:
def load_deepfake_real_images_dataset(data_root):
    """
    Carrega o dataset Deepfake and Real Images.
    Este dataset tem frames REAIS de vídeos deepfake!
    """
    data_root = Path(data_root)
    
    # Encontrar estrutura
    possible_paths = [
        data_root / 'Dataset',
        data_root / 'dataset',
        data_root,
    ]
    
    dataset_root = None
    for p in possible_paths:
        if (p / 'Train').exists() or (p / 'train').exists():
            dataset_root = p
            break
    
    if dataset_root is None:
        print('Estrutura do diretório:')
        for item in data_root.rglob('*'):
            if item.is_dir():
                print(f'  {item}')
        raise ValueError(f'Não encontrei a pasta Train em {data_root}')
    
    # Carregar paths
    train_dir = dataset_root / 'Train' if (dataset_root / 'Train').exists() else dataset_root / 'train'
    val_dir = dataset_root / 'Validation' if (dataset_root / 'Validation').exists() else dataset_root / 'validation'
    
    def get_images(folder):
        exts = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
        images = []
        for ext in exts:
            images.extend(folder.glob(ext))
        return images
    
    # Train
    train_fake_dir = train_dir / 'Fake' if (train_dir / 'Fake').exists() else train_dir / 'fake'
    train_real_dir = train_dir / 'Real' if (train_dir / 'Real').exists() else train_dir / 'real'
    
    train_fake = get_images(train_fake_dir)
    train_real = get_images(train_real_dir)
    
    # Validation
    if val_dir.exists():
        val_fake_dir = val_dir / 'Fake' if (val_dir / 'Fake').exists() else val_dir / 'fake'
        val_real_dir = val_dir / 'Real' if (val_dir / 'Real').exists() else val_dir / 'real'
        
        val_fake = get_images(val_fake_dir)
        val_real = get_images(val_real_dir)
    else:
        val_fake, val_real = [], []
    
    print(f'Dataset Deepfake and Real Images:')
    print(f'  Train: {len(train_real)} real, {len(train_fake)} fake')
    print(f'  Val:   {len(val_real)} real, {len(val_fake)} fake')
    
    # Combinar
    all_paths = list(train_real) + list(train_fake) + list(val_real) + list(val_fake)
    all_labels = [0]*len(train_real) + [1]*len(train_fake) + [0]*len(val_real) + [1]*len(val_fake)
    
    return all_paths, all_labels


def load_custom_data(fake_dir, real_dir):
    """Carrega seus próprios deepfakes e imagens reais."""
    fake_dir = Path(fake_dir)
    real_dir = Path(real_dir)
    
    exts = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    
    fake_images = []
    real_images = []
    
    for ext in exts:
        fake_images.extend(fake_dir.glob(ext))
        real_images.extend(real_dir.glob(ext))
    
    print(f'\nDados customizados:')
    print(f'  Fake: {len(fake_images)}')
    print(f'  Real: {len(real_images)}')
    
    paths = list(real_images) + list(fake_images)
    labels = [0]*len(real_images) + [1]*len(fake_images)
    
    return paths, labels

In [ ]:
# Carregar dataset principal
try:
    main_paths, main_labels = load_deepfake_real_images_dataset('/content/data_dfr')
    print(f'\nTotal principal: {len(main_paths)} imagens')
except Exception as e:
    print(f'Erro ao carregar dataset: {e}')
    print('\nUsando dataset 140k como fallback...')
    # Fallback para 140k se o outro não estiver disponível
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data_140k --unzip
    
    data_root = Path('/content/data_140k/real_vs_fake/real-vs-fake')
    train_real = list((data_root / 'train' / 'real').glob('*.jpg'))
    train_fake = list((data_root / 'train' / 'fake').glob('*.jpg'))
    
    main_paths = train_real + train_fake
    main_labels = [0]*len(train_real) + [1]*len(train_fake)
    print(f'Fallback: {len(main_paths)} imagens (140k dataset)')

In [ ]:
# Carregar dados customizados (se existirem)
custom_paths, custom_labels = load_custom_data(config.CUSTOM_FAKE_DIR, config.CUSTOM_REAL_DIR)

# Combinar tudo
all_paths = main_paths + custom_paths
all_labels = main_labels + custom_labels

print(f'\nTotal combinado: {len(all_paths)} imagens')
print(f'Fake: {sum(all_labels)} ({sum(all_labels)/len(all_labels)*100:.1f}%)')
print(f'Real: {len(all_labels) - sum(all_labels)} ({(1-sum(all_labels)/len(all_labels))*100:.1f}%)')

## 5. Upload de Seus Próprios Deepfakes!

**IMPORTANTE:** Para o modelo generalizar para SEUS deepfakes, adicione-os ao treino!

In [ ]:
# Fazer upload de deepfakes que VOCÊ criou
# O modelo vai aprender os padrões específicos deles!

print('📤 Faça upload de seus deepfakes para a pasta custom_fakes')
print('📤 E fotos reais para a pasta custom_reals')
print('')
print('No painel esquerdo do Colab:')
print('1. Navegue até /content/custom_fakes')
print('2. Clique com botão direito → Upload')
print('3. Adicione seus deepfakes')
print('')
print('Repita para /content/custom_reals com fotos reais')

In [ ]:
# Ou faça upload programaticamente
from google.colab import files

# Descomente para fazer upload:
# print('Selecione seus deepfakes:')
# uploaded_fakes = files.upload()
# for name, data in uploaded_fakes.items():
#     with open(config.CUSTOM_FAKE_DIR / name, 'wb') as f:
#         f.write(data)

# print('\nSelecione fotos reais:')
# uploaded_reals = files.upload()
# for name, data in uploaded_reals.items():
#     with open(config.CUSTOM_REAL_DIR / name, 'wb') as f:
#         f.write(data)

In [ ]:
# Recarregar dados após upload
custom_paths, custom_labels = load_custom_data(config.CUSTOM_FAKE_DIR, config.CUSTOM_REAL_DIR)

# Replicar dados customizados 10x para dar mais peso no treino
if len(custom_paths) > 0:
    CUSTOM_WEIGHT = 10  # Quantas vezes replicar
    custom_paths_weighted = custom_paths * CUSTOM_WEIGHT
    custom_labels_weighted = custom_labels * CUSTOM_WEIGHT
    print(f'\nDados customizados replicados {CUSTOM_WEIGHT}x: {len(custom_paths_weighted)} imagens')
else:
    custom_paths_weighted = []
    custom_labels_weighted = []

# Recombinar
all_paths = main_paths + custom_paths_weighted
all_labels = main_labels + custom_labels_weighted

print(f'\nTotal final: {len(all_paths)} imagens')

## 6. Visualizar Amostras

In [ ]:
# Visualizar algumas imagens para verificar qualidade
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Reais
real_idxs = [i for i, l in enumerate(all_labels) if l == 0]
for i, idx in enumerate(random.sample(real_idxs, min(5, len(real_idxs)))):
    img = Image.open(all_paths[idx])
    axes[0, i].imshow(img)
    axes[0, i].axis('off')
    axes[0, i].set_title('REAL', color='green')

# Fakes
fake_idxs = [i for i, l in enumerate(all_labels) if l == 1]
for i, idx in enumerate(random.sample(fake_idxs, min(5, len(fake_idxs)))):
    img = Image.open(all_paths[idx])
    axes[1, i].imshow(img)
    axes[1, i].axis('off')
    axes[1, i].set_title('FAKE', color='red')

plt.tight_layout()
plt.show()

## 7. Split Train/Val/Test

In [ ]:
# Split stratificado
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.15,
    stratify=all_labels,
    random_state=42
)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=0.1,
    stratify=train_labels,
    random_state=42
)

print(f'Train: {len(train_paths)} ({sum(train_labels)/len(train_labels)*100:.1f}% fake)')
print(f'Val:   {len(val_paths)} ({sum(val_labels)/len(val_labels)*100:.1f}% fake)')
print(f'Test:  {len(test_paths)} ({sum(test_labels)/len(test_labels)*100:.1f}% fake)')

## 8. Loss e Augmentations

In [ ]:
class FocalLossWithAlpha(nn.Module):
    def __init__(self, gamma=3.0, alpha=0.8, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        if self.label_smoothing > 0:
            targets_smooth = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        else:
            targets_smooth = targets
        
        bce = F.binary_cross_entropy_with_logits(logits, targets_smooth, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        alpha_weight = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_weight * focal_weight * bce
        return loss.mean()


def get_train_transforms():
    return A.Compose([
        A.Resize(config.IMG_SIZE, config.IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.2, rotate_limit=20, p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1),
        ], p=0.7),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1),
            A.ISONoise(p=1),
        ], p=0.3),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1),
            A.MotionBlur(blur_limit=5, p=1),
        ], p=0.3),
        A.ImageCompression(quality_lower=40, quality_upper=95, p=0.5),
        A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_valid_transforms():
    return A.Compose([
        A.Resize(config.IMG_SIZE, config.IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

## 9. Dataset e DataLoader

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = [str(p) for p in image_paths]
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        try:
            image = np.array(Image.open(self.image_paths[idx]).convert('RGB'))
        except:
            image = np.zeros((config.IMG_SIZE, config.IMG_SIZE, 3), dtype=np.uint8)
        
        if self.transform:
            image = self.transform(image=image)['image']
        
        return image, torch.tensor(self.labels[idx], dtype=torch.float32)


# Criar datasets
train_dataset = DeepfakeDataset(train_paths, train_labels, transform=get_train_transforms())
val_dataset = DeepfakeDataset(val_paths, val_labels, transform=get_valid_transforms())
test_dataset = DeepfakeDataset(test_paths, test_labels, transform=get_valid_transforms())

# Weighted sampler
class_counts = Counter(train_labels)
class_weights = {cls: len(train_labels) / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(sample_weights, len(train_labels), replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, sampler=sampler,
                          num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE*2, shuffle=False,
                        num_workers=config.NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE*2, shuffle=False,
                         num_workers=config.NUM_WORKERS, pin_memory=True)

print(f'Batches: {len(train_loader)} train, {len(val_loader)} val, {len(test_loader)} test')

## 10. Modelo

In [ ]:
class DeepfakeDetectorV4(nn.Module):
    def __init__(self, model_name='efficientnet_b0', pretrained=True, dropout=0.5):
        super().__init__()
        
        self.backbone = timm.create_model(model_name, pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        num_features = self.backbone.num_features
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 1)
        )
        
        self.register_buffer('best_threshold', torch.tensor(0.30))
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features).squeeze(-1)


model = DeepfakeDetectorV4(config.MODEL_NAME, config.PRETRAINED).to(device)
print(f'Parâmetros: {sum(p.numel() for p in model.parameters()):,}')

## 11. Treino

In [ ]:
criterion = FocalLossWithAlpha(gamma=config.FOCAL_GAMMA, alpha=config.FOCAL_ALPHA,
                                label_smoothing=config.LABEL_SMOOTHING)

optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE,
                        weight_decay=config.WEIGHT_DECAY)

# Scheduler com warmup
num_training_steps = len(train_loader) * config.EPOCHS
num_warmup_steps = len(train_loader) * config.WARMUP_EPOCHS

def lr_lambda(step):
    if step < num_warmup_steps:
        return float(step) / float(max(1, num_warmup_steps))
    progress = float(step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler(enabled=config.USE_AMP)

print(f'Focal Loss: gamma={config.FOCAL_GAMMA}, alpha={config.FOCAL_ALPHA}')

In [ ]:
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam


def train_epoch(model, loader, criterion, optimizer, scheduler, scaler, epoch):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    pbar = tqdm(loader, desc=f'Epoch {epoch}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Mixup
        if config.USE_MIXUP and np.random.random() > 0.3:
            images, y_a, y_b, lam = mixup_data(images, labels, config.MIXUP_ALPHA)
            use_mixup = True
        else:
            use_mixup = False
        
        optimizer.zero_grad()
        
        with autocast(enabled=config.USE_AMP):
            outputs = model(images)
            if use_mixup:
                loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
            else:
                loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        
        if not use_mixup:
            with torch.no_grad():
                all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': total_loss / (pbar.n + 1)})
    
    avg_loss = total_loss / len(loader)
    auc = roc_auc_score(all_labels, all_preds) if len(all_preds) > 0 else 0
    return avg_loss, auc


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for images, labels in tqdm(loader, desc='Validating'):
        images, labels = images.to(device), labels.to(device)
        
        with autocast(enabled=config.USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        total_loss += loss.item()
        all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(loader)
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    auc = roc_auc_score(all_labels, all_preds)
    
    return avg_loss, auc, all_preds, all_labels

In [ ]:
def find_best_threshold(labels, preds, metric='f2'):
    best_thresh, best_score = 0.5, 0
    
    for thresh in np.arange(config.MIN_THRESHOLD, config.MAX_THRESHOLD, 0.01):
        preds_bin = (preds >= thresh).astype(int)
        score = fbeta_score(labels, preds_bin, beta=2, zero_division=0)
        if score > best_score:
            best_score = score
            best_thresh = thresh
    
    # Métricas
    preds_bin = (preds >= best_thresh).astype(int)
    tn = ((labels == 0) & (preds_bin == 0)).sum()
    tp = ((labels == 1) & (preds_bin == 1)).sum()
    fn = ((labels == 1) & (preds_bin == 0)).sum()
    fp = ((labels == 0) & (preds_bin == 1)).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    print(f'\nThreshold: {best_thresh:.3f}')
    print(f'  Recall (detectar fakes): {recall:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  F2-score: {best_score:.4f}')
    
    return best_thresh

In [ ]:
# TREINO
best_auc = 0
patience_counter = 0

for epoch in range(1, config.EPOCHS + 1):
    print(f'\n{"="*50}')
    print(f'Epoch {epoch}/{config.EPOCHS} | LR: {optimizer.param_groups[0]["lr"]:.2e}')
    
    train_loss, train_auc = train_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, epoch)
    val_loss, val_auc, val_preds, val_labels = validate(model, val_loader, criterion)
    
    print(f'Train - Loss: {train_loss:.4f}, AUC: {train_auc:.4f}')
    print(f'Val   - Loss: {val_loss:.4f}, AUC: {val_auc:.4f}')
    
    if val_auc > best_auc:
        best_auc = val_auc
        patience_counter = 0
        
        best_thresh = find_best_threshold(val_labels, val_preds)
        model.best_threshold = torch.tensor(best_thresh)
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_auc': best_auc,
            'best_threshold': best_thresh,
            'config': {'model_name': config.MODEL_NAME, 'img_size': config.IMG_SIZE}
        }, config.OUTPUT_DIR / f'best_model_{config.MODEL_NAME}_v4.pth')
        
        print(f'*** Melhor modelo salvo! AUC: {val_auc:.4f} ***')
    else:
        patience_counter += 1
        if patience_counter >= config.PATIENCE:
            print(f'\nEarly stopping na epoch {epoch}')
            break

print(f'\nMelhor AUC: {best_auc:.4f}')

## 12. Avaliação Final

In [ ]:
# Carregar melhor modelo
checkpoint = torch.load(config.OUTPUT_DIR / f'best_model_{config.MODEL_NAME}_v4.pth')
model.load_state_dict(checkpoint['model_state_dict'])
best_thresh = checkpoint['best_threshold']

# Avaliar no teste
test_loss, test_auc, test_preds, test_labels = validate(model, test_loader, criterion)

print(f'\n{"="*50}')
print('RESULTADOS NO TESTE')
print(f'AUC: {test_auc:.4f}')

final_thresh = find_best_threshold(test_labels, test_preds)

In [ ]:
# Confusion matrix
test_preds_bin = (test_preds >= final_thresh).astype(int)

cm = confusion_matrix(test_labels, test_preds_bin)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title(f'Confusion Matrix (threshold={final_thresh:.3f})')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.show()

print(classification_report(test_labels, test_preds_bin, target_names=['Real', 'Fake']))

In [ ]:
# Salvar modelo final
model.best_threshold = torch.tensor(final_thresh)

torch.save({
    'model_state_dict': model.state_dict(),
    'best_auc': test_auc,
    'best_threshold': final_thresh,
    'config': {'model_name': config.MODEL_NAME, 'img_size': config.IMG_SIZE}
}, config.OUTPUT_DIR / f'final_model_{config.MODEL_NAME}_v4.pth')

print(f'\nModelo salvo com threshold {final_thresh:.3f}')

## 13. Download

In [ ]:
from google.colab import files

files.download(str(config.OUTPUT_DIR / f'final_model_{config.MODEL_NAME}_v4.pth'))

print(f'\nModelo baixado!')
print(f'Threshold: {final_thresh:.3f}')
print(f'AUC: {test_auc:.4f}')

## Resumo

### Por que v4 é diferente:

1. **Dataset correto** - Usa frames de VÍDEOS deepfake, não rostos sintéticos
2. **Seus dados** - Adiciona seus próprios deepfakes ao treino (10x replicados)
3. **Alpha = 0.8** - 4x mais peso para detectar fakes
4. **Threshold baixo (~0.25-0.35)** - Prefere falsos positivos

### Para usar na sua aplicação:

```python
# Copiar modelo para:
# model/weights/efficientnet_b0_deepfake_v4.pth
```